# 06 · Work IQ MCP 로 취합 → 위키 Markdown 생성

**Work IQ MCP 파이프라인 2/2** — 자연어 질의 한 줄로 Work IQ에서 관련 지식을 취합하고,
LLM으로 위키용 Markdown 초안을 생성합니다. `04_nl_aggregate_to_md`와 **같은 흐름**
(**extract → generate → save/commit**)이며, 소스가 Work IQ MCP(단일 엔드포인트, 범용 도구)라는
점만 다릅니다.

추출은 두 방식을 제공합니다:
- **기본 `extract_via_ask`** — Work IQ `ask`(M365 Copilot)가 메일·Teams를 가로질러 취합(간단·안정).
- **선택 `extract_via_agent`** — 04식 에이전트 루프. LLM이 Work IQ **읽기 전용** 도구를 직접 호출.

출력은 노트북 전용 `notebook/wiki/`에 저장됩니다.

## 셋업 (설정 + 인증 + LLM + 취합/생성)

In [ ]:
# ============================================================
# 셋업 — 이 셀 하나로 설정·인증 준비 (pipeline/*.py 없이 독립 실행)
# Work IQ MCP: 단일 엔드포인트 + 범용 10개 도구(fetch/ask/search_paths/...).
#
# [인증 — 기본은 웹앱/노트북01~04와 "통일"]
#   기본(통일): 자체 앱 등록(CLIENT_ID) + device-code + 공유 캐시(.token_cache.json).
#              → Mail/Teams(01~04)와 같은 앱·같은 로그인·같은 캐시를 재사용 → "로그인 1번".
#              → 앱 등록에 Work IQ 위임 권한 'WorkIQAgent.Ask' 동의가 되어 있어야 함(웹앱과 동일).
#   옵션(공개): Microsoft 공개 Work IQ 클라이언트(ba081686...) + 별도 캐시(.workiq.token_cache.json).
#              → 앱 등록 없이 바로 시도할 때. WORKIQ_USE_PUBLIC_CLIENT=true 로 켠다.
#   (두 방식 모두 회사·학교 계정 + 관리자 동의 + EULA 필요, 개인 계정 불가)
# ============================================================
import os, json, asyncio
from pathlib import Path
from contextlib import AsyncExitStack

import msal
from dotenv import load_dotenv
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client, create_mcp_http_client

# 1) 프로젝트 루트를 찾아 .env 로드 (.env.example/requirements.txt가 있는 폴더)
def find_project_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / '.env.example').is_file() or (base / 'requirements.txt').is_file():
            return base
    return Path.cwd()

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env')

# 2) 공통 설정 (모두 .env로 덮어쓸 수 있음)
TENANT_ID  = (os.environ.get('TENANT_ID') or '').strip() or 'organizations'
WORKIQ_URL = (os.environ.get('WORKIQ_MCP_SERVER_URL') or '').strip() or 'https://workiq.svc.cloud.microsoft/mcp'
SCOPE      = (os.environ.get('WORKIQ_SCOPE') or '').strip() or 'fdcc1f02-fc51-4226-8753-f668596af7f7/WorkIQAgent.Ask'
AUTHORITY  = f'https://login.microsoftonline.com/{TENANT_ID}'
SCOPES     = [SCOPE]
REDIRECT_PORT = int((os.environ.get('WORKIQ_REDIRECT_PORT') or '12798').strip())

# 2-1) 인증 방식 토글 — 기본은 '자체 앱(통일)', 옵션으로 'MS 공개 클라이언트'.
USE_PUBLIC_CLIENT = (os.environ.get('WORKIQ_USE_PUBLIC_CLIENT') or 'false').strip().lower() in ('1', 'true', 'yes')

if USE_PUBLIC_CLIENT:
    # [옵션] Microsoft 공개 Work IQ 클라이언트 — 앱 등록 불필요, Work IQ 전용 별도 캐시.
    CLIENT_ID   = (os.environ.get('WORKIQ_CLIENT_ID') or '').strip() or 'ba081686-5d24-4bc6-a0d6-d034ecffed87'
    AUTH_MODE   = (os.environ.get('WORKIQ_AUTH_MODE') or 'interactive').strip()   # interactive(기본) | device_code
    TOKEN_CACHE = PROJECT_ROOT / ((os.environ.get('WORKIQ_TOKEN_CACHE_PATH') or '').strip() or '.workiq.token_cache.json')
else:
    # [기본] 웹앱/01~04와 동일한 자체 앱 등록 + device-code + 공유 캐시(로그인 1회를 공유).
    CLIENT_ID   = (os.environ.get('CLIENT_ID') or '').strip()
    AUTH_MODE   = (os.environ.get('AUTH_MODE') or 'device_code').strip()          # device_code(기본) | interactive
    TOKEN_CACHE = PROJECT_ROOT / ((os.environ.get('TOKEN_CACHE_PATH') or '').strip() or '.token_cache.json')

# 3) MSAL 인증 — Work IQ 위임(delegated) 토큰. 최초 1회 로그인/동의 후 캐시 재사용.
#    로그인 흐름은 AUTH_MODE로만 분기(두 방식 공통):
#      device_code : 원격/헤드리스 커널용. https://microsoft.com/devicelogin 에 코드 입력.
#      interactive : 로컬 브라우저 로그인(루프백).
_cache = msal.SerializableTokenCache()
if TOKEN_CACHE.exists():
    _cache.deserialize(TOKEN_CACHE.read_text(encoding='utf-8'))

def _save_cache():
    if _cache.has_state_changed:
        TOKEN_CACHE.write_text(_cache.serialize(), encoding='utf-8')

def ensure_token() -> str:
    """Work IQ access token 확보 (캐시 우선, 없으면 로그인).
    - 기본(자체 앱): .env의 CLIENT_ID 필요. Mail/Teams와 같은 앱·캐시라 이미 로그인돼 있으면 재사용.
    - 옵션(공개)  : WORKIQ_USE_PUBLIC_CLIENT=true → CLIENT_ID 기본값(ba081686...) 사용, 앱 등록 불필요.
    """
    assert CLIENT_ID, ('CLIENT_ID가 필요합니다. 기본(통일)은 .env의 CLIENT_ID를 설정하고, '
                       '앱 등록 없이 쓰려면 WORKIQ_USE_PUBLIC_CLIENT=true 로 공개 클라이언트를 사용하세요.')
    app = msal.PublicClientApplication(CLIENT_ID, authority=AUTHORITY, token_cache=_cache)
    accounts = app.get_accounts()
    res = app.acquire_token_silent(SCOPES, account=accounts[0]) if accounts else None
    if not (res and 'access_token' in res):
        if AUTH_MODE == 'device_code':                 # 원격/헤드리스
            flow = app.initiate_device_flow(scopes=SCOPES)
            print(flow['message'])                      # 인증 URL + 코드 안내
            res = app.acquire_token_by_device_flow(flow)
        else:                                           # 로컬 브라우저 로그인
            res = app.acquire_token_interactive(scopes=SCOPES, port=REDIRECT_PORT)
    _save_cache()
    if not res or 'access_token' not in res:
        raise RuntimeError((res or {}).get('error_description', 'access token 획득 실패'))
    return res['access_token']

print('PROJECT_ROOT:', PROJECT_ROOT)
print('WORKIQ_URL  :', WORKIQ_URL)
print('AUTH 방식    :', 'MS 공개 클라이언트(옵션)' if USE_PUBLIC_CLIENT else '자체 앱(통일·기본)')
print('AUTHORITY   :', AUTHORITY, '| CLIENT_ID set:', bool(CLIENT_ID), '| AUTH_MODE:', AUTH_MODE)
print('SCOPE       :', SCOPE)
print('TOKEN_CACHE :', TOKEN_CACHE.name)

In [ ]:
# --- MCP 헬퍼: 단일 Work IQ 엔드포인트. 연결은 호출마다 열고 항상 닫는다 ---
async def _with_session(token, fn):
    # 최신 MCP SDK: 인증 헤더는 httpx 클라이언트에 담아 streamable_http_client에 전달
    async with create_mcp_http_client(headers={'Authorization': f'Bearer {token}'}) as http_client:
        async with streamable_http_client(WORKIQ_URL, http_client=http_client) as (r, w, _):
            async with ClientSession(r, w) as s:       # MCP 세션 초기화(handshake)
                await s.initialize()
                return await fn(s)

async def list_tools(token):
    res = await _with_session(token, lambda s: s.list_tools())
    return list(res.tools or [])

async def call_tool(token, name, args=None):
    return await _with_session(token, lambda s: s.call_tool(name, arguments=args or {}))

def tool_schema(tool):
    sc = getattr(tool, 'inputSchema', None) or {'type': 'object', 'properties': {}}
    return sc.model_dump() if hasattr(sc, 'model_dump') else sc

def tool_payload(result):
    """Work IQ 결과를 dict로: structuredContent 우선, 없으면 text 블록을 JSON 파싱."""
    sc = getattr(result, 'structuredContent', None)
    if isinstance(sc, dict) and sc:
        return sc
    for b in (getattr(result, 'content', None) or []):
        if getattr(b, 'type', None) == 'text' and getattr(b, 'text', ''):
            try:
                return json.loads(b.text)
            except Exception:
                pass
    return None

def content_to_text(result) -> str:
    """content 블록을 평탄화 (\\uXXXX 이스케이프 JSON은 한글로 복원)."""
    parts = []
    for b in (getattr(result, 'content', None) or []):
        if getattr(b, 'type', None) == 'text':
            t = getattr(b, 'text', '')
            try:
                t = json.dumps(json.loads(t), ensure_ascii=False, indent=2)
            except Exception:
                pass
            parts.append(t)
        else:
            data = b.model_dump() if hasattr(b, 'model_dump') else b
            parts.append('```json\n' + json.dumps(data, ensure_ascii=False, indent=2, default=str) + '\n```')
    text = '\n'.join(parts)
    return f'ERROR: {text}' if getattr(result, 'isError', False) else text

def readable(result) -> str:
    """사람이 보기 좋게: ask→response, search_paths→paths, fetch→results[].data(.value), 그 외 JSON."""
    if getattr(result, 'isError', False):
        return 'ERROR: ' + content_to_text(result)
    p = tool_payload(result)
    if isinstance(p, dict):
        if 'response' in p:                                  # ask
            return p.get('response') or '(빈 응답)'
        if 'paths' in p:                                     # search_paths
            return '\n'.join(f" - {x.get('path')}  [{', '.join(x.get('operations', []))}]"
                             for x in (p.get('paths') or []))
        if 'results' in p:                                   # fetch
            out = []
            for r in (p.get('results') or []):
                data = r.get('data') if isinstance(r, dict) else r
                sc = r.get('statusCode') if isinstance(r, dict) else None
                body = data.get('value') if isinstance(data, dict) and 'value' in data else data
                out.append((f'[statusCode={sc}]\n' if sc else '') +
                           json.dumps(body, ensure_ascii=False, indent=2))
            return '\n'.join(out)
        return json.dumps(p, ensure_ascii=False, indent=2)
    return content_to_text(result)

In [ ]:
# --- LLM + (읽기 전용) 에이전트 루프: Work IQ 범용 도구를 LLM에 노출 ---
# 안전: 메일/메시지 본문은 신뢰할 수 없는 입력(프롬프트 인젝션 가능)이므로
#       쓰기 도구(create/update/delete/do_action)는 절대 노출하지 않는다.
READ_ONLY_TOOLS = {'ask', 'fetch', 'search_paths', 'get_schema', 'list_agents', 'call_function'}

def make_openai():
    """Azure OpenAI(Entra 키리스=az login/Managed Identity 우선, 없으면 API 키) 또는 OpenAI 클라이언트."""
    ep = (os.environ.get('AZURE_OPENAI_ENDPOINT') or '').strip()
    dep = (os.environ.get('AZURE_OPENAI_DEPLOYMENT') or '').strip()
    if ep and dep:
        from openai import AzureOpenAI
        common = dict(azure_endpoint=ep, azure_deployment=dep,
                      api_version=os.environ.get('AZURE_OPENAI_API_VERSION', '2024-10-21'))
        key = (os.environ.get('AZURE_OPENAI_API_KEY') or '').strip()
        if key:
            return {'client': AzureOpenAI(api_key=key, **common), 'model': dep}
        # 빈 AZURE_OPENAI_API_KEY("")가 환경에 남으면 키리스가 막히므로 제거
        os.environ.pop('AZURE_OPENAI_API_KEY', None)
        from azure.identity import DefaultAzureCredential, get_bearer_token_provider
        tp = get_bearer_token_provider(DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default')
        return {'client': AzureOpenAI(azure_ad_token_provider=tp, **common), 'model': dep}
    okey = (os.environ.get('OPENAI_API_KEY') or '').strip()
    if okey:
        from openai import OpenAI
        return {'client': OpenAI(api_key=okey), 'model': os.environ.get('OPENAI_MODEL', 'gpt-4o-mini')}
    raise RuntimeError('LLM 미설정 — .env에 AZURE_OPENAI_* 또는 OPENAI_API_KEY를 넣으세요.')

async def run_agent(token, user_message, max_iters=6):
    """LLM이 Work IQ 읽기 도구를 스스로 호출하며 자료를 모으는 루프 (04의 에이전트와 동일한 흐름)."""
    oa = make_openai()
    trace = []
    async with create_mcp_http_client(headers={'Authorization': f'Bearer {token}'}) as http_client:
        async with streamable_http_client(WORKIQ_URL, http_client=http_client) as (r, w, _):
            async with ClientSession(r, w) as s:
                await s.initialize()
                all_tools = (await s.list_tools()).tools or []
                tools = [{'type': 'function', 'function': {
                            'name': t.name,
                            'description': (t.description or t.name),
                            'parameters': tool_schema(t)}}
                         for t in all_tools if t.name in READ_ONLY_TOOLS]     # 읽기 도구만
                messages = [
                    {'role': 'system', 'content':
                        'You are connected to the Microsoft "Work IQ" MCP server. Tools act on relative '
                        'Microsoft Graph paths. Use search_paths to discover paths, get_schema to learn '
                        'shapes, fetch to read entities (use $select/$top; /me/chats returns chats, read '
                        'messages via /me/chats/{id}/messages), and ask for cross-source semantic search. '
                        'Never invent IDs or paths; resolve them first. You are READ-ONLY: never create, '
                        'update, or delete. Answer in the user language, concisely.'},
                    {'role': 'user', 'content': user_message}]
                for _ in range(max_iters):
                    comp = await asyncio.to_thread(
                        oa['client'].chat.completions.create,
                        model=oa['model'], messages=messages, tools=tools, tool_choice='auto', temperature=0)
                    msg = comp.choices[0].message
                    m = {'role': 'assistant', 'content': msg.content or ''}
                    if msg.tool_calls:
                        m['tool_calls'] = [{'id': c.id, 'type': 'function', 'function': {
                            'name': c.function.name, 'arguments': c.function.arguments}} for c in msg.tool_calls]
                    messages.append(m)
                    if not msg.tool_calls:
                        return {'answer': msg.content or '(no content)', 'trace': trace}
                    for c in msg.tool_calls:
                        try:
                            args = json.loads(c.function.arguments or '{}')
                        except Exception:
                            args = {}
                        name = c.function.name
                        if name not in READ_ONLY_TOOLS:                       # 방어적 차단
                            out = f'ERROR: tool {name} is not allowed (read-only).'
                        else:
                            try:
                                out = content_to_text(await s.call_tool(name, arguments=args))
                            except Exception as exc:
                                out = f'ERROR calling {name}: {exc}'
                        trace.append({'tool': name, 'args': args, 'result': out})
                        messages.append({'role': 'tool', 'tool_call_id': c.id, 'content': out[:8000]})
                return {'answer': '도구 호출 한도(max_iters) 초과.', 'trace': trace}

In [ ]:
# --- 추출(extract) + 위키 생성(generate) + 저장/커밋 헬퍼 ---
import re, subprocess
from datetime import date, datetime, timezone, timedelta

WIKI_DIR  = PROJECT_ROOT / 'notebook' / 'wiki'      # 노트북 전용 출력 폴더
CONNECTOR = 'Work IQ MCP'

_WRITER_PROMPT = (
    '너는 사내 엔지니어링 지식 위키를 관리하는 시니어 테크니컬 라이터다. 수집된 원본(메일/Teams)을 '
    '하나의 잘 구조화된 Markdown 위키 문서로 정리해라.\n'
    '- 첫 줄은 H1 제목 한 줄: `# <간결한 제목>`\n'
    '- 원본 언어(한국어면 한국어)로 작성하고 H2/H3로 개요/방법/예시/주의사항 등 구성\n'
    '- 명령어·코드·설정은 코드블록에 원문 그대로 보존, 잡담·인사·불필요한 PII는 제거\n'
    '- 사실을 지어내지 말 것. 마지막에 `## 출처` 섹션을 두되, 자료에 **실제로 있는** '
    '소스/작성자/날짜/ID만 적고 없으면 "미상"으로 표기\n'
    '- 문서 Markdown만 출력 (앞뒤 설명이나 전체 코드펜스 없이)')

def default_date_range(days=7):
    """최근 days일 범위 (start, end) ISO 날짜."""
    end = date.today(); start = end - timedelta(days=max(days, 0))
    return start.isoformat(), end.isoformat()

def build_extraction_prompt(query, start, end):
    return (f'지식 위키용 자료를 수집한다. 아래 주제와 관련되고 재사용 가능한 기술 지식/노하우'
            f'(how-to, 결정사항, 트러블슈팅, 팁, 설정, 교훈)를 담은 메일/Teams 메시지만 모아라.\n\n'
            f'주제:\n{query}\n\n기간(포함): {start} .. {end}\n\n'
            '메일과 Teams를 모두 살피고, 주제와 무관한 잡담·일정은 버려라. 남긴 항목마다 '
            '[출처(메일/Teams)/작성자/시각/제목/실제 ID]와 유용한 내용 요약을 정리하고(명령·코드는 원문 인용), '
            '관련 자료가 없으면 없다고 명시하라. 반드시 실제로 확인된 메타데이터만 적고 지어내지 마라. '
            '대화체가 아니라 위키 원자료로 정확하게 소스별로 묶어 반환.')

async def extract_via_ask(token, query, start, end):
    """기본 추출: Work IQ `ask`(M365 Copilot)로 메일·Teams를 가로질러 취합."""
    res = await call_tool(token, 'ask',
                          {'question': build_extraction_prompt(query, start, end), 'timeZone': 'Asia/Seoul'})
    p = tool_payload(res)
    return (p or {}).get('response') or content_to_text(res)

async def extract_via_agent(token, query, start, end):
    """(선택) 04식 에이전트 루프: LLM이 Work IQ 읽기 도구를 직접 호출."""
    return (await run_agent(token, build_extraction_prompt(query, start, end)))['answer']

async def to_markdown(query, start, end, material):
    """수집한 원자료(material)를 위키 Markdown 문서로 생성 (저장/커밋은 안 함)."""
    oa = make_openai()
    user = (f'추출 질의: {query}\n범위: {start} .. {end}\n커넥터: {CONNECTOR}\n\n'
            f'=== 수집된 원본 자료 ===\n{material}')
    comp = await asyncio.to_thread(
        oa['client'].chat.completions.create, model=oa['model'],
        messages=[{'role': 'system', 'content': _WRITER_PROMPT}, {'role': 'user', 'content': user}],
        temperature=0.2)
    body = comp.choices[0].message.content or ''
    title = next((l[2:].strip() for l in body.splitlines() if l.startswith('# ')), query or '제목 없음')
    slug = re.sub(r'-{2,}', '-', re.sub(r'[^\w가-힣]+', '-', title.lower(), flags=re.UNICODE)).strip('-') or 'untitled'
    fm = ('---\n'
          f'title: "{title}"\n'
          f'generated: "{datetime.now(timezone.utc).isoformat(timespec="seconds")}"\n'
          f'connector: "{CONNECTOR}"\n'
          'sources: ["Microsoft 365 (Mail, Teams)"]\n'
          f'date_range: "{start}..{end}"\n'
          f'query: "{query}"\n'
          'generator: "llmwiki-notebook-workiq"\n'
          '---')
    return {'title': title, 'slug': slug, 'markdown': f'{fm}\n\n{body.strip()}\n'}

def save_doc(markdown, slug):
    WIKI_DIR.mkdir(parents=True, exist_ok=True)
    path = WIKI_DIR / f'{date.today().isoformat()}-{slug}.md'
    path.write_text(markdown, encoding='utf-8')
    return path

def save_and_commit(markdown, slug, message=None):
    """문서를 저장하고 그 파일만 git 커밋 (pathspec 제한)."""
    path = save_doc(markdown, slug)
    def git(*a):
        return subprocess.run(['git', *a], cwd=str(PROJECT_ROOT), capture_output=True, text=True)
    git('add', '--', str(path))
    c = git('commit', '-m', message or f'docs(wiki): add {slug}', '--', str(path))
    return {'path': str(path), 'commit': (c.stdout + c.stderr).strip() or '(변경 없음/실패)'}

def list_docs():
    if not WIKI_DIR.exists():
        return []
    return [{'filename': p.name, 'path': str(p)} for p in sorted(WIKI_DIR.glob('*.md'), reverse=True)]

## 1. 토큰 확보 + 추출 범위 지정

In [ ]:
token = ensure_token()
make_openai()   # LLM 설정 확인

QUERY = 'Postgres 튜닝, Kubernetes 트러블슈팅, CI 캐시 등 재사용 가능한 기술 노하우'
START, END = default_date_range(days=7)   # 최근 7일 (넓히려면 days 증가)
print('질의:', QUERY)
print('범위:', START, '..', END)

## 2. 추출 (extract) — 기본: `ask` 기반 취합
Work IQ `ask`가 메일·Teams를 가로질러 주제·기간에 맞는 지식을 모읍니다. (`ask`는 응답까지
수십 초 걸릴 수 있습니다.)

In [ ]:
extract_material = await extract_via_ask(token, QUERY, START, END)
material = extract_material
print(material[:2000])

### 2-b. (선택) 에이전트 루프로 추출
04처럼 **LLM이 Work IQ 읽기 도구를 직접 호출**하게 하려면 아래 셀을 실행하세요.
(쓰기 도구는 노출되지 않습니다.) 결과를 `material`로 대체합니다.

In [ ]:
# material = await extract_via_agent(token, QUERY, START, END)
# print(material[:2000])

## 3. 생성 (generate) — 위키 Markdown 초안 만들기 (저장 안 함)

In [ ]:
doc = await to_markdown(QUERY, START, END, material)
print('제목:', doc['title'])
print('slug:', doc['slug'])
markdown_text = doc['markdown']

## 4. 초안 미리보기

In [ ]:
from IPython.display import Markdown, display
import re
# 저장본에는 YAML front matter가 들어가지만, 미리보기에서는 본문만 렌더링
_preview = re.sub(r'\A\ufeff?---\r?\n.*?\r?\n---\r?\n+', '', markdown_text, count=1, flags=re.DOTALL)
display(Markdown(_preview))

## 5. 검토·수정
아래 `markdown_text` 문자열을 직접 편집하세요.

In [ ]:
# 예: markdown_text = markdown_text.replace('오타', '수정')
print(markdown_text[:800])

## 6. 저장 + 커밋 → `notebook/wiki/`
검토가 끝나면 저장/커밋합니다. 커밋은 **해당 문서 파일만** 포함합니다(pathspec 제한).
저장만 하려면 `save_doc(...)`을 쓰세요.

In [ ]:
DO_COMMIT = False   # 실제로 커밋하려면 True

if DO_COMMIT:
    out = save_and_commit(markdown_text, doc['slug'])
    print('저장:', out['path'])
    print('커밋:', out['commit'])
else:
    path = save_doc(markdown_text, doc['slug'])
    print('저장만 완료(커밋 안 함):', path)

## 7. 저장된 위키 문서 목록

In [ ]:
for d in list_docs():
    print(f"- {d['filename']}")

🎉 완료! Work IQ MCP로 취합·생성한 Markdown이 `notebook/wiki/`에 저장됩니다.